In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("hospital_final_dataset.csv")

df.head()

,Admission_ID,Patient_ID,Doctor_ID,Department_ID,Admission_Date,Discharge_Date,Admission_Type,Diagnosis,Ward_Type,Room_No,...,Lab_Cost,Other_Charges,Total_Amount,Insurance_Covered,Patient_Payable,Payment_Status,Age,Readmission,Days_To_Readmission,Readmission_Date
0,A000001,P01079,D0026,102,3/1/2026,2026-03-03,Referral,Migraine,General,G-217,...,5428,4614,32894,26380.12,6513.88,Paid,31,No,0,Not Readmitted
1,A000002,P00881,D0101,105,12/23/2025,2025-12-29,Emergency,Kidney Stone,Semi-Private,SP-862,...,5745,2992,37176,0.00,37176.00,Paid,77,No,0,Not Readmitted
2,A000003,P02567,D0109,105,4/6/2025,2025-04-10,Referral,Diabetes,General,G-994,...,9122,1726,35873,29913.95,5959.05,Paid,90,No,0,Not Readmitted
3,A000004,P00972,D0088,105,6/25/2025,2025-06-28,Emergency,Diabetes,General,G-995,...,5167,4781,39865,0.00,39865.00,Paid,40,No,0,Not Readmitted
4,A000005,P03546,D0030,102,12/25/2024,2024-12-28,Elective,Migraine,General,G-924,...,8915,1938,34635,25962.94,8672.06,Paid,50,No,0,Not Readmitted


In [3]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

df.info()

Rows : 10000
Columns : 45
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 45 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Admission_ID         10000 non-null  object 
 1   Patient_ID           10000 non-null  object 
 2   Doctor_ID            10000 non-null  object 
 3   Department_ID        10000 non-null  int64  
 4   Admission_Date       10000 non-null  object 
 5   Discharge_Date       10000 non-null  object 
 6   Admission_Type       10000 non-null  object 
 7   Diagnosis            10000 non-null  object 
 8   Ward_Type            10000 non-null  object 
 9   Room_No              10000 non-null  object 
 10  Length_of_Stay       10000 non-null  int64  
 11  Status               10000 non-null  object 
 12  Patient_First_Name   10000 non-null  object 
 13  Patient_Last_Name    10000 non-null  object 
 14  Patient_Gender       10000 non-null  object 
 15  DOB        

In [5]:
# Convert date columns to datetime
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"])
df["Discharge_Date"] = pd.to_datetime(df["Discharge_Date"])


In [6]:
print(df.dtypes[["Admission_Date", "Discharge_Date"]])

Admission_Date    datetime64[ns]
Discharge_Date    datetime64[ns]
dtype: object


In [7]:
# KPI 1 - Total Admissions
total_admissions = df["Admission_ID"].nunique()

print("="*40)
print("KPI 1 : Total Admissions")
print("="*40)
print(f"Total Admissions : {total_admissions}")

KPI 1 : Total Admissions
Total Admissions : 10000


In [8]:
# KPI 2 - Average Length of Stay

average_los = df["Length_of_Stay"].mean()

print("="*40)
print("KPI 2 : Average Length of Stay")
print("="*40)
print(f"Average Length of Stay : {average_los:.2f} Days")

KPI 2 : Average Length of Stay
Average Length of Stay : 6.96 Days


In [9]:
# KPI 3 - Readmission Rate

readmitted = (df["Readmission"] == "Yes").sum()

readmission_rate = (readmitted / total_admissions) * 100

print("="*40)
print("KPI 3 : Readmission Rate")
print("="*40)
print(f"Readmitted Patients : {readmitted}")
print(f"Readmission Rate : {readmission_rate:.2f}%")

KPI 3 : Readmission Rate
Readmitted Patients : 1491
Readmission Rate : 14.91%


In [10]:
df.groupby("Department_Name")["Total_Beds"].unique()

Department_Name
Cardiology           [60]
Emergency            [80]
General Medicine    [120]
Neurology            [45]
Oncology             [40]
Orthopedics          [55]
Surgery              [70]
Name: Total_Beds, dtype: object

In [11]:
# KPI 4 : Occupancy Rate

# Total hospital beds
total_hospital_beds = (
    df.groupby("Department_Name")["Total_Beds"]
      .first()
      .sum()
)

# Total patient bed days
patient_bed_days = df["Length_of_Stay"].sum()

# Study period
study_period_days = (
    df["Discharge_Date"].max()
    - df["Admission_Date"].min()
).days + 1

# Total available bed days
available_bed_days = (
    total_hospital_beds *
    study_period_days
)

# Occupancy Rate
occupancy_rate = (
    patient_bed_days /
    available_bed_days
) * 100

print("=" * 45)
print("KPI 4 : Occupancy Rate")
print("=" * 45)
print(f"Total Hospital Beds      : {total_hospital_beds}")
print(f"Patient Bed Days         : {patient_bed_days}")
print(f"Study Period             : {study_period_days} Days")
print(f"Available Bed Days       : {available_bed_days}")
print(f"Occupancy Rate           : {occupancy_rate:.2f}%")

KPI 4 : Occupancy Rate
Total Hospital Beds      : 470
Patient Bed Days         : 69610
Study Period             : 754 Days
Available Bed Days       : 354380
Occupancy Rate           : 19.64%


In [12]:
# KPI 5 : Bed Utilization Rate

bed_utilization_rate = (
    total_admissions /
    total_hospital_beds
)

print("=" * 45)
print("KPI 5 : Bed Utilization Rate")
print("=" * 45)
print(f"Total Admissions     : {total_admissions}")
print(f"Total Hospital Beds  : {total_hospital_beds}")
print(f"Bed Utilization Rate : {bed_utilization_rate:.2f} Admissions per Bed")

KPI 5 : Bed Utilization Rate
Total Admissions     : 10000
Total Hospital Beds  : 470
Bed Utilization Rate : 21.28 Admissions per Bed


In [6]:
# Calculate the raw efficiency score
raw_score = (
    department_efficiency["Total_Admissions"] /
    (
        department_efficiency["Total_Beds"] *
        department_efficiency["Average_LOS"]
    )
)

# Normalize to percentage
department_efficiency["Efficiency_Percentage"] = (
    raw_score / raw_score.max() * 100
).round(2)

# Round Average LOS
department_efficiency["Average_LOS"] = (
    department_efficiency["Average_LOS"].round(2)
)

# Sort by Efficiency Percentage
department_efficiency = department_efficiency.sort_values(
    by="Efficiency_Percentage",
    ascending=False
)

# Keep only the required columns
department_efficiency = department_efficiency[
    [
        "Total_Admissions",
        "Average_LOS",
        "Total_Beds",
        "Efficiency_Percentage"
    ]
]

department_efficiency

,Total_Admissions,Average_LOS,Total_Beds,Efficiency_Percentage
Department_Name,,,,
General Medicine,3336,4.25,120,100.00
Neurology,1603,6.46,45,84.19
Cardiology,1641,6.10,60,68.54
Surgery,882,4.99,70,38.57
Orthopedics,830,8.45,55,27.28
Emergency,833,7.45,80,21.36
Oncology,875,19.95,40,16.75


In [14]:
# ============================================
# Final KPI Summary
# ============================================

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Admissions",
        "Average Length of Stay (Days)",
        "Readmission Rate (%)",
        "Occupancy Rate (%)",
        "Bed Utilization (Admissions per Bed)"
    ],
    "Value": [
        total_admissions,
        round(average_los, 2),
        round(readmission_rate, 2),
        round(occupancy_rate, 2),
        round(bed_utilization_rate, 2)
    ]
})

print("=" * 50)
print("HOSPITAL KPI SUMMARY")
print("=" * 50)
display(kpi_summary)

HOSPITAL KPI SUMMARY


,KPI,Value
0,Total Admissions,10000.00
1,Average Length of Stay (Days),6.96
2,Readmission Rate (%),14.91
3,Occupancy Rate (%),19.64
4,Bed Utilization (Admissions per Bed),21.28


In [15]:
kpi_summary.to_csv("hospital_kpi_summary.csv", index=False)

In [8]:
department_efficiency.to_csv("department_efficiency.csv")